# PipelineWatch-NG — Module 1: Data Ingestion
### Sentinel-1 SAR + FIRMS/VIIRS + TROPOMI SO2 · Niger Delta

**All plots save to `outputs/` and display inline — no kernel crash.**

---

In [ ]:
import ee, os, json, gc
import pandas as pd
import numpy as np
from datetime import datetime
print("Imports OK.")


In [ ]:
import matplotlib
matplotlib.use("Agg")  # saves PNG to disk — no GUI thread — no kernel crash
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import Image, display
import gc, os

OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def show(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    gc.collect()
    display(Image(path))
    print("Saved: " + path)

print("Backend: " + matplotlib.get_backend())
print("All plots -> outputs/ and displayed inline.")


## Cell 3 — GEE authentication

In [ ]:
GEE_PROJECT_ID = "pipelinewatch-ng"
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("GEE connected: " + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("Authenticated.")


## Cell 4 — Config

In [ ]:
ROI_BOUNDS = [6.50, 5.00, 7.20, 5.80]
ROI_CENTER = [5.40, 6.85]
ROI_ZOOM   = 9
BASELINE_START = "2023-01-01"; BASELINE_END = "2023-06-30"
RECENT_START   = "2024-01-01"; RECENT_END   = "2024-06-30"
SAR_DARK_SPOT_SIGMA = 1.5
FIRMS_BRIGHTNESS_K  = 330.0
SO2_THRESHOLD_DU    = 3.0
CACHE_DIR = "../data/cached"
os.makedirs(CACHE_DIR, exist_ok=True)
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)
print("Config ready.")


## Cell 5 — SAR functions

In [ ]:
def get_s1_collection(roi, start, end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi).filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
        .select(["VV", "VH"])
    )

def compute_sar_features(image, roi, sigma=1.5):
    vv = image.select("VV"); vh = image.select("VH")
    stats = vv.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=roi, scale=100, maxPixels=1e9, bestEffort=True)
    mean_vv = ee.Number(stats.get("VV_mean"))
    std_vv  = ee.Number(stats.get("VV_stdDev"))
    dark_mask = vv.lt(mean_vv.subtract(std_vv.multiply(sigma))).rename("dark_spot_mask")
    dark_mag  = vv.subtract(mean_vv).multiply(-1).divide(std_vv).rename("dark_spot_magnitude")
    ratio     = ee.Image(10).pow(vv.divide(10)).divide(ee.Image(10).pow(vh.divide(10))).rename("vv_vh_ratio")
    return image.addBands(dark_mask).addBands(dark_mag).addBands(ratio)

def build_s1_composite(roi, start, end, sigma=1.5):
    col   = get_s1_collection(roi, start, end)
    count = col.size().getInfo()
    print("  S1 scenes (" + start + " to " + end + "): " + str(count))
    if count == 0: return None, None, 0
    col = col.map(lambda img: compute_sar_features(img, roi, sigma))
    return col.median().clip(roi), col, count

print("SAR functions ready.")


## Cell 6 — FIRMS + TROPOMI functions

In [ ]:
def get_firms_collection(roi, start, end):
    return (ee.ImageCollection("FIRMS").filterDate(start, end)
            .filterBounds(roi).select(["T21","confidence","line_number"]))

def compute_firms_composite(col, roi):
    return (col.select("T21").max().rename("T21_max").clip(roi)
            .addBands(col.select("T21").count().rename("fire_count").clip(roi))
            .addBands(col.select("T21").mean().rename("T21_mean").clip(roi)))

def extract_fire_hotspots(comp, roi, t21_k=330.0, min_count=3):
    hot = comp.select("T21_max").gt(t21_k).And(comp.select("fire_count").gte(min_count))
    vecs = comp.select("T21_max").updateMask(hot).toInt().reduceToVectors(
        geometry=roi, scale=375, geometryType="centroid",
        eightConnected=True, maxPixels=1e9, bestEffort=True)
    def ann(f):
        g = f.geometry()
        s = comp.reduceRegion(reducer=ee.Reducer.mean().combine(ee.Reducer.max(),sharedInputs=True),
                              geometry=g.buffer(500), scale=375, maxPixels=1e6)
        t = ee.Number(s.get("T21_max_max"))
        c = ee.Number(s.get("fire_count_mean"))
        src = ee.Algorithms.If(t.gt(380),"flare_candidate",
              ee.Algorithms.If(c.gte(5),"refinery_candidate","agricultural_or_other"))
        return f.set({"T21_max_K":t,"fire_count":c,"signal_type":"FIRMS_fire_hotspot",
                      "likely_source":src,"confidence":ee.Algorithms.If(c.gte(5),"high","nominal")})
    return vecs.map(ann)

def get_tropomi_collection(roi, start, end):
    return (ee.ImageCollection("COPERNICUS/S5P/NRTI/L3_SO2")
            .filterDate(start, end).filterBounds(roi)
            .select(["SO2_column_number_density","cloud_fraction"]))

def compute_so2_composite(col, roi, max_cloud=0.3):
    def mc(img):
        return img.select("SO2_column_number_density").multiply(2241.5).rename("SO2_DU")\
                  .updateMask(img.select("cloud_fraction").lt(max_cloud))
    c = col.map(mc)
    return (c.mean().rename("SO2_mean_DU").clip(roi)
            .addBands(c.max().rename("SO2_max_DU").clip(roi))
            .addBands(c.count().rename("SO2_obs_count").clip(roi)))

def extract_so2_anomalies(comp, roi, threshold=3.0, min_obs=5):
    mask = comp.select("SO2_mean_DU").gt(threshold).And(comp.select("SO2_obs_count").gte(min_obs))
    vecs = comp.select("SO2_mean_DU").updateMask(mask).multiply(100).toInt().reduceToVectors(
        geometry=roi, scale=5500, geometryType="polygon",
        eightConnected=True, maxPixels=1e9, bestEffort=True)
    def ann(f):
        g = f.geometry()
        s = comp.reduceRegion(reducer=ee.Reducer.mean().combine(ee.Reducer.max(),sharedInputs=True),
                              geometry=g, scale=5500, maxPixels=1e6)
        m = ee.Number(s.get("SO2_mean_DU_mean"))
        conf = ee.Algorithms.If(m.gt(8),"high",ee.Algorithms.If(m.gt(5),"nominal","low"))
        return f.set({"SO2_mean_DU":m,"SO2_max_DU":ee.Number(s.get("SO2_max_DU_max")),
                      "signal_type":"TROPOMI_SO2_anomaly","confidence":conf})
    return vecs.map(ann)

def compute_fire_gas_risk(so2, firms, roi, so2_thr=3.0, t21_thr=330.0):
    return (so2.select("SO2_mean_DU").gt(so2_thr).reproject(crs="EPSG:4326",scale=5500).multiply(2)
            .add(firms.select("T21_max").gt(t21_thr).reproject(crs="EPSG:4326",scale=5500))
            .rename("fire_gas_risk_score").clip(roi))

print("FIRMS + TROPOMI functions ready.")


## Cell 7 — Run SAR ingestion

In [ ]:
print("=== Sentinel-1 SAR ===")
s1_baseline, _, n_baseline = build_s1_composite(ROI, BASELINE_START, BASELINE_END)
s1_recent,   _, n_recent   = build_s1_composite(ROI, RECENT_START,   RECENT_END)
n_spots = "deferred to Module 2"
print("Baseline: " + str(n_baseline) + " scenes | Recent: " + str(n_recent) + " scenes")


## Cell 8 — Run FIRMS ingestion

In [ ]:
print("=== FIRMS / VIIRS ===")
firms_base_col  = get_firms_collection(ROI, BASELINE_START, BASELINE_END)
n_firms_base    = firms_base_col.size().getInfo()
firms_base_comp = compute_firms_composite(firms_base_col, ROI)
firms_rec_col   = get_firms_collection(ROI, RECENT_START, RECENT_END)
n_firms_rec     = firms_rec_col.size().getInfo()
firms_rec_comp  = compute_firms_composite(firms_rec_col, ROI)
fire_hotspots   = extract_fire_hotspots(firms_rec_comp, ROI, t21_k=FIRMS_BRIGHTNESS_K, min_count=3)
n_hotspots      = fire_hotspots.size().getInfo()
print("Baseline images: " + str(n_firms_base))
print("Recent images  : " + str(n_firms_rec))
print("Fire hotspots  : " + str(n_hotspots))


## Cell 9 — Run TROPOMI ingestion

In [ ]:
print("=== TROPOMI SO2 ===")
trop_base_col = get_tropomi_collection(ROI, BASELINE_START, BASELINE_END)
n_trop_base   = trop_base_col.size().getInfo()
so2_baseline  = compute_so2_composite(trop_base_col, ROI)
trop_rec_col  = get_tropomi_collection(ROI, RECENT_START, RECENT_END)
n_trop_rec    = trop_rec_col.size().getInfo()
so2_recent    = compute_so2_composite(trop_rec_col, ROI)
so2_anomalies = extract_so2_anomalies(so2_recent, ROI, threshold=SO2_THRESHOLD_DU)
n_so2         = so2_anomalies.size().getInfo()
fire_gas_score = compute_fire_gas_risk(so2_recent, firms_rec_comp, ROI)
print("Baseline images: " + str(n_trop_base))
print("Recent images  : " + str(n_trop_rec))
print("SO2 anomalies  : " + str(n_so2))
print("Risk score ready.")


## Cell 10 — Summary table

In [ ]:
import pandas as pd
df = pd.DataFrame([
    {"Sensor":"Sentinel-1 SAR","Scenes":n_recent,"Detections":n_spots,"All-weather":"Yes","Night":"Yes"},
    {"Sensor":"FIRMS/VIIRS","Scenes":n_firms_rec,"Detections":n_hotspots,"All-weather":"Partial","Night":"Yes"},
    {"Sensor":"TROPOMI SO2","Scenes":n_trop_rec,"Detections":n_so2,"All-weather":"Partial","Night":"No"},
])
print("=== Module 1 Ingestion Summary ===")
print(df.to_string(index=False))


## Cell 11 — SO2 statistics bar chart

In [ ]:
import numpy as np

def get_so2_stats(comp, roi):
    return comp.select("SO2_mean_DU").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(),sharedInputs=True)
                          .combine(ee.Reducer.percentile([50,75]),sharedInputs=True),
        geometry=roi, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()

gc.collect()
base_s = get_so2_stats(so2_baseline, ROI)
rec_s  = get_so2_stats(so2_recent, ROI)

metrics = ["Mean","Median","p75","Max"]
keys    = ["SO2_mean_DU_mean","SO2_mean_DU_p50","SO2_mean_DU_p75","SO2_mean_DU_max"]
bvals   = [base_s.get(k,0) or 0 for k in keys]
rvals   = [rec_s.get(k,0)  or 0 for k in keys]

x, w = np.arange(len(metrics)), 0.35
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x-w/2, bvals, w, label="Baseline 2023", color="#378ADD", alpha=0.85)
ax.bar(x+w/2, rvals, w, label="Recent 2024",   color="#E24B4A", alpha=0.85)
ax.axhline(SO2_THRESHOLD_DU, color="#854F0B", linewidth=1.5, linestyle="--",
           label="Anomaly threshold (" + str(SO2_THRESHOLD_DU) + " DU)")
ax.set_ylabel("SO2 column density (Dobson Units)")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title("PipelineWatch-NG — TROPOMI SO2 Baseline vs Recent\nTrans Niger Pipeline corridor")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.03, str(round(h,2)),
                ha="center", va="bottom", fontsize=8)
show("m1_so2_comparison.png")


## Cell 12 — Monthly fire hotspot bar chart

In [ ]:
months = [
    ("Jan","2023-01-01","2023-02-01"),("Feb","2023-02-01","2023-03-01"),
    ("Mar","2023-03-01","2023-04-01"),("Apr","2023-04-01","2023-05-01"),
    ("May","2023-05-01","2023-06-01"),("Jun","2023-06-01","2023-07-01"),
]
months_rec = [
    ("Jan","2024-01-01","2024-02-01"),("Feb","2024-02-01","2024-03-01"),
    ("Mar","2024-03-01","2024-04-01"),("Apr","2024-04-01","2024-05-01"),
    ("May","2024-05-01","2024-06-01"),("Jun","2024-06-01","2024-07-01"),
]

def count_month(col, start, end, thr):
    mc = col.filterDate(start, end)
    if mc.size().getInfo() == 0: return 0
    hot = mc.select("T21").max().gt(thr)
    r = hot.reduceRegion(reducer=ee.Reducer.sum(),geometry=ROI,
                         scale=375,maxPixels=1e9,bestEffort=True).get("T21")
    return ee.Number(r).getInfo() or 0

print("Counting monthly fire pixels...")
base_counts = [count_month(firms_base_col,s,e,FIRMS_BRIGHTNESS_K) for _,s,e in months]
rec_counts  = [count_month(firms_rec_col, s,e,FIRMS_BRIGHTNESS_K) for _,s,e in months_rec]

labels = [m[0] for m in months]
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-w/2, base_counts, w, label="Baseline 2023", color="#378ADD", alpha=0.85)
ax.bar(x+w/2, rec_counts,  w, label="Recent 2024",   color="#E24B4A", alpha=0.85)
ax.set_ylabel("Hot pixels (T21 > 330K)")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("PipelineWatch-NG — Monthly VIIRS Fire Hotspot Count\nTrans Niger Pipeline corridor")
ax.legend(); ax.grid(axis="y", alpha=0.3)
show("m1_fire_monthly.png")


## Cell 13 — Export GeoJSON outputs

In [ ]:
from datetime import datetime
os.makedirs(CACHE_DIR, exist_ok=True)

def export_fc(fc, filename, label, max_features=500):
    path = os.path.join(CACHE_DIR, filename)
    n = fc.size().getInfo()
    if n == 0:
        with open(path,"w") as f: json.dump({"type":"FeatureCollection","features":[]},f)
        print("  " + label + ": 0 features")
        return 0
    gj = fc.limit(max_features).getInfo()
    gj["metadata"] = {"exported":datetime.now().isoformat(),"count":n,"signal":label}
    with open(path,"w") as f: json.dump(gj,f,indent=2)
    print("  " + label + ": " + str(n) + " -> " + path)
    return n

print("=== Exporting GeoJSON ===")
sar_sampled = (s1_recent.select(["VV","dark_spot_mask","dark_spot_magnitude"])
               .sample(region=ROI,scale=200,numPixels=300,geometries=True)
               .filter(ee.Filter.eq("dark_spot_mask",1)))
export_fc(sar_sampled,   "sar_dark_spots.geojson",  "SAR dark spots")
export_fc(fire_hotspots, "fire_hotspots.geojson",   "VIIRS fire hotspots")
export_fc(so2_anomalies, "so2_anomalies.geojson",   "TROPOMI SO2 anomalies")

meta = {"module":"M1_ingestion","exported":datetime.now().isoformat(),
        "scene_counts":{"s1_baseline":n_baseline,"s1_recent":n_recent,
                        "firms_baseline":n_firms_base,"firms_recent":n_firms_rec,
                        "tropomi_baseline":n_trop_base,"tropomi_recent":n_trop_rec},
        "detections":{"fire_hotspots":n_hotspots,"so2_anomalies":n_so2}}
with open(os.path.join(CACHE_DIR,"m1_metadata.json"),"w") as f: json.dump(meta,f,indent=2)
print("Metadata saved.")
print()
print("Module 1 complete. Push data/cached/ to GitHub.")
